# PySpark & la MLlib

## Récupération des données

In [1]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh titanic
!ls -lh titanic/

remote: Enumerating objects: 2, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 2 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Réception d'objets: 100% (2/2), 2.96 Kio | 2.96 Mio/s, fait.
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 5 (delta 0), reused 5 (delta 0), pack-reused 0 (from 0)
Réception d'objets: 100% (5/5), 33.71 Kio | 1.98 Mio/s, fait.
✓ titanic — 104K
total 100K
-rw-rw-r-- 1 mog mog  237 11 nov.  22:28 data_description.txt
-rw-rw-r-- 1 mog mog 3,2K 11 nov.  22:28 gender_submission.csv
-rw-rw-r-- 1 mog mog  709 11 nov.  22:28 README.md
-rw-rw-r-- 1 mog mog  28K 11 nov.  22:28 test.csv
-rw-rw-r-- 1 mog mog  60K 11 nov.  22:28 train.csv


## Import des bibliothèques nécessaires

In [2]:
!pip install pyspark

pyenv: pip: command not found

The `pip' command exists in these Python versions:
  3.12.11
  3.13.7

Note: See 'pyenv help global' for tips on allowing both
      python2 and python3 to be found.


In [3]:
import re
from pathlib import Path

import pyspark.ml.feature as MF
import pyspark.sql.functions as F
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.classification import (
  GBTClassifier,
  LinearSVC,
  LogisticRegression,
  RandomForestClassifier,
)
from pyspark.ml.evaluation import (
  BinaryClassificationEvaluator,
  MulticlassClassificationEvaluator,
)
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql import SparkSession

## Chargement des données

Le fichier CSV est chargé dans un `DataFrame` Spark.

In [4]:
dossier_donnees = Path("titanic")
chemin_csv = dossier_donnees / "train.csv"

In [5]:
spark = SparkSession.builder.appName("Démonstration de PySpark").getOrCreate()
df = spark.read.csv(str(chemin_csv), header=True, inferSchema=True)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/11 22:28:38 WARN Utils: Your hostname, pug, resolves to a loopback address: 127.0.1.1; using 192.168.1.94 instead (on interface wlo1)
26/11/11 22:28:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/11 22:28:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

In [6]:
nb_passagers = df.count()
print(nb_passagers)

891


### Partitions

Un `DataFrame` est découpé en partitions, que les exécuteurs traitent en parallèle, une tâche par partition. Ici, tout tourne dans un seul processus local, dont les cœurs tiennent le rôle des exécuteurs : `defaultParallelism` en donne le nombre. Le fichier, petit, tient dans une seule partition.

In [7]:
print(f"Partitions : {df.rdd.getNumPartitions()}")
print(f"Parallélisme par défaut : {spark.sparkContext.defaultParallelism}")

Partitions : 1
Parallélisme par défaut : 8


In [8]:
df.show(5)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|
+-----------+--------+------+--------------------+------+----+-----+-----+------

In [9]:
df.describe().show()

26/11/11 22:28:52 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 8:>                                                          (0 + 1) / 1]

+-------+-----------------+-------------------+------------------+--------------------+------+------------------+------------------+-------------------+------------------+-----------------+-----+--------+
|summary|      PassengerId|           Survived|            Pclass|                Name|   Sex|               Age|             SibSp|              Parch|            Ticket|             Fare|Cabin|Embarked|
+-------+-----------------+-------------------+------------------+--------------------+------+------------------+------------------+-------------------+------------------+-----------------+-----+--------+
|  count|              891|                891|               891|                 891|   891|               714|               891|                891|               891|              891|  204|     889|
|   mean|            446.0| 0.3838383838383838| 2.308641975308642|                NULL|  NULL| 29.69911764705882|0.5230078563411896|0.38159371492704824|260318.54916792738| 32.20420

In [10]:
df.printSchema()

root
 |-- PassengerId: integer (nullable = true)
 |-- Survived: integer (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- SibSp: integer (nullable = true)
 |-- Parch: integer (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: double (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)



## Renommage des colonnes en *snake case*, la convention de Python

In [11]:
df = df.toDF(*(re.sub(r"(?<!^)(?=[A-Z])", "_", c).lower() for c in df.columns))
df.show(5)

+------------+--------+------+--------------------+------+----+------+-----+----------------+-------+-----+--------+
|passenger_id|survived|pclass|                name|   sex| age|sib_sp|parch|          ticket|   fare|cabin|embarked|
+------------+--------+------+--------------------+------+----+------+-----+----------------+-------+-----+--------+
|           1|       0|     3|Braund, Mr. Owen ...|  male|22.0|     1|    0|       A/5 21171|   7.25| NULL|       S|
|           2|       1|     1|Cumings, Mrs. Joh...|female|38.0|     1|    0|        PC 17599|71.2833|  C85|       C|
|           3|       1|     3|Heikkinen, Miss. ...|female|26.0|     0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|           4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|     1|    0|          113803|   53.1| C123|       S|
|           5|       0|     3|Allen, Mr. Willia...|  male|35.0|     0|    0|          373450|   8.05| NULL|       S|
+------------+--------+------+--------------------+------+----+-

## Exploration des données

In [12]:
df.select("survived", "pclass", "embarked").show(5)

+--------+------+--------+
|survived|pclass|embarked|
+--------+------+--------+
|       0|     3|       S|
|       1|     1|       C|
|       1|     3|       S|
|       1|     1|       S|
|       0|     3|       S|
+--------+------+--------+
only showing top 5 rows


In [13]:
df.groupBy("survived").count().show()

+--------+-----+
|survived|count|
+--------+-----+
|       1|  342|
|       0|  549|
+--------+-----+



In [14]:
df.groupBy("sex", "survived").count().show()

+------+--------+-----+
|   sex|survived|count|
+------+--------+-----+
|  male|       0|  468|
|female|       1|  233|
|female|       0|   81|
|  male|       1|  109|
+------+--------+-----+



Le même décompte en SQL : une vue temporaire donne un nom au `DataFrame`, que la requête interroge. Les deux écritures produisent le même plan d'exécution.

In [15]:
df.createOrReplaceTempView("passagers")
spark.sql(
  "SELECT sex, survived, COUNT(*) AS count FROM passagers GROUP BY sex, survived"
).show()

+------+--------+-----+
|   sex|survived|count|
+------+--------+-----+
|  male|       0|  468|
|female|       1|  233|
|female|       0|   81|
|  male|       1|  109|
+------+--------+-----+



In [16]:
df.groupBy("pclass", "survived").count().show()

+------+--------+-----+
|pclass|survived|count|
+------+--------+-----+
|     1|       0|   80|
|     3|       1|  119|
|     1|       1|  136|
|     2|       1|   87|
|     2|       0|   97|
|     3|       0|  372|
+------+--------+-----+



## Préparation des données

### Valeurs nulles

In [17]:
df.select(
  [F.count(F.when(F.col(colonne).isNull(), 1)).alias(colonne) for colonne in df.columns]
).show()

+------------+--------+------+----+---+---+------+-----+------+----+-----+--------+
|passenger_id|survived|pclass|name|sex|age|sib_sp|parch|ticket|fare|cabin|embarked|
+------------+--------+------+----+---+---+------+-----+------+----+-----+--------+
|           0|       0|     0|   0|  0|177|     0|    0|     0|   0|  687|       2|
+------------+--------+------+----+---+---+------+-----+------+----+-----+--------+



### Analyse de l'âge des passagers

In [18]:
age_moyen = df.select(F.avg("age")).first()[0]
print(age_moyen)

29.69911764705882


On pourrait utiliser cet âge moyen pour l'imputation des âges inconnus, mais nous allons voir comment aller un peu plus loin et calculer l'âge moyen par civilité.

In [19]:
df = df.withColumn("civilite", F.regexp_extract(F.col("name"), r"([A-Za-z]+)\.", 1))
df.select("name", "sex", "civilite").show(5, truncate=False)

+---------------------------------------------------+------+--------+
|name                                               |sex   |civilite|
+---------------------------------------------------+------+--------+
|Braund, Mr. Owen Harris                            |male  |Mr      |
|Cumings, Mrs. John Bradley (Florence Briggs Thayer)|female|Mrs     |
|Heikkinen, Miss. Laina                             |female|Miss    |
|Futrelle, Mrs. Jacques Heath (Lily May Peel)       |female|Mrs     |
|Allen, Mr. William Henry                           |male  |Mr      |
+---------------------------------------------------+------+--------+
only showing top 5 rows


In [20]:
df.select("civilite").distinct().show()

+--------+
|civilite|
+--------+
|     Don|
|    Miss|
|Countess|
|     Col|
|     Rev|
|    Lady|
|  Master|
|     Mme|
|    Capt|
|      Mr|
|      Dr|
|     Mrs|
|     Sir|
|Jonkheer|
|    Mlle|
|   Major|
|      Ms|
+--------+



Les civilités rares sont regroupées. « Dr » désigne des hommes et une femme : il est rattaché à « Mr » ou à « Mrs » selon le sexe.

In [21]:
regroupements = dict(
  Miss=["Mlle", "Ms"],
  Mr=["Major", "Capt", "Sir", "Don"],
  Mrs=["Lady", "Countess", "Mme"],
  Autre=["Jonkheer", "Col", "Rev"],
)
df = df.replace(
  {nom: civilite for civilite, noms in regroupements.items() for nom in noms},
  subset=["civilite"],
)
df = df.withColumn(
  "civilite",
  F.when(
    F.col("civilite") == "Dr",
    F.when(F.col("sex") == "female", "Mrs").otherwise("Mr"),
  ).otherwise(F.col("civilite")),
)

In [22]:
df.groupBy("civilite").count().show()

+--------+-----+
|civilite|count|
+--------+-----+
|    Miss|  185|
|  Master|   40|
|      Mr|  528|
|     Mrs|  129|
|   Autre|    9|
+--------+-----+



### Ajout d'une nouvelle caractéristique

La taille de la famille.

In [23]:
df = df.withColumn("taille_famille", F.col("sib_sp") + F.col("parch") + 1)

In [24]:
df.groupBy("taille_famille").count().show()

+--------------+-----+
|taille_famille|count|
+--------------+-----+
|             1|  537|
|             6|   22|
|             3|  102|
|             5|   15|
|             4|   29|
|             8|    6|
|             7|   12|
|            11|    7|
|             2|  161|
+--------------+-----+



In [25]:
df = df.withColumn("seul", (F.col("taille_famille") == 1).cast("long"))
df.select("sib_sp", "parch", "taille_famille", "seul").show(5)

+------+-----+--------------+----+
|sib_sp|parch|taille_famille|seul|
+------+-----+--------------+----+
|     1|    0|             2|   0|
|     1|    0|             2|   0|
|     0|    0|             1|   1|
|     1|    0|             2|   0|
|     0|    0|             1|   1|
+------+-----+--------------+----+
only showing top 5 rows


### Transformations et actions

Aucune des cellules précédentes qui modifient `df` n'a lu le fichier : `withColumn`, `replace` ou `select` sont des *transformations*, qui ne font qu'ajouter une étape au plan d'exécution. Seules les *actions*, comme `show`, `count` ou `collect`, lancent un calcul, et Spark optimise alors tout le plan d'un coup. `explain` affiche le plan physique qu'il a choisi : la lecture du CSV (`FileScan csv`), puis les colonnes calculées (`Project`), toutes marquées `*(1)` : Spark les exécute en un seul passage sur les données.

In [26]:
df.explain()

== Physical Plan ==
*(1) Project [passenger_id#929, survived#930, pclass#931, name#932, sex#933, age#934, sib_sp#935, parch#936, ticket#937, fare#938, cabin#939, embarked#940, civilite#1250, taille_famille#1277, cast((taille_famille#1277 = 1) as bigint) AS seul#1305L]
+- *(1) Project [passenger_id#929, survived#930, pclass#931, name#932, sex#933, age#934, sib_sp#935, parch#936, ticket#937, fare#938, cabin#939, embarked#940, CASE WHEN (civilite#1249 = Dr) THEN CASE WHEN (sex#933 = female) THEN Mrs ELSE Mr END ELSE civilite#1249 END AS civilite#1250, ((sib_sp#935 + parch#936) + 1) AS taille_famille#1277]
   +- *(1) Project [passenger_id#929, survived#930, pclass#931, name#932, sex#933, age#934, sib_sp#935, parch#936, ticket#937, fare#938, cabin#939, embarked#940, CASE WHEN (civilite#1231 = Don) THEN Mr WHEN (civilite#1231 = Col) THEN Autre WHEN (civilite#1231 = Mme) THEN Mrs WHEN (civilite#1231 = Sir) THEN Mr WHEN (civilite#1231 = Jonkheer) THEN Autre WHEN (civilite#1231 = Ms) THEN Miss 

### Séparation en ensembles d'apprentissage et de test

Les colonnes calculées jusqu'ici ne dépendent que de la ligne du passager. La suite apprend des données : l'âge moyen par civilité, le port le plus fréquent, puis les catégories que les encodeurs retiennent. On sépare donc les données maintenant, et tout ce qui s'apprend le sera sur l'ensemble d'apprentissage seul : calculé sur les 891 passagers, il ferait entrer dans le modèle une information sur ceux du test (une fuite de données).

In [27]:
entrainement, test = df.randomSplit([0.8, 0.2], seed=11)
print(f"Apprentissage : {entrainement.count()}, test : {test.count()}")

Apprentissage : 703, test : 188


### Âge moyen par civilité

In [28]:
ages_moyens = entrainement.groupBy("civilite").agg(F.avg("age").alias("age_moyen"))
ages_moyens.show()

+--------+------------------+
|civilite|         age_moyen|
+--------+------------------+
|    Miss|21.743589743589745|
|  Master|4.5125925925925925|
|      Mr|32.458730158730155|
|     Mrs| 35.54455445544554|
|   Autre| 42.42857142857143|
+--------+------------------+



### Port d'embarquement le plus fréquent

La caractéristique *embarked* (le port d'embarquement) n'a que deux valeurs manquantes : on leur attribue le port le plus fréquent de l'ensemble d'apprentissage.

In [29]:
entrainement.groupBy("embarked").count().show()

+--------+-----+
|embarked|count|
+--------+-----+
|       Q|   60|
|    NULL|    2|
|       C|  137|
|       S|  504|
+--------+-----+



In [30]:
port_frequent = (
  entrainement.dropna(subset=["embarked"])
  .groupBy("embarked")
  .count()
  .orderBy(F.desc("count"))
  .first()["embarked"]
)
print(port_frequent)

S


### Imputation

Chaque passager d'âge inconnu reçoit l'âge moyen des passagers de l'apprentissage qui partagent sa civilité. La jointure `left` garde tous les passagers, même d'une civilité absente de l'apprentissage, et `coalesce` ne remplace que les âges nuls.

La caractéristique *cabin* est supprimée : le numéro de cabine est quasi unique pour chaque passager qui est en cabine, et provoquerait rapidement du surapprentissage.

In [31]:
def completer(donnees):
  return (
    donnees.join(ages_moyens, "civilite", "left")
    .withColumn("age", F.coalesce("age", "age_moyen"))
    .drop("age_moyen", "cabin")
    .fillna({"embarked": port_frequent})
  )


entrainement = completer(entrainement)
test = completer(test)
entrainement.select("name", "civilite", "age", "embarked").show(5)

+--------------------+--------+----+--------+
|                name|civilite| age|embarked|
+--------------------+--------+----+--------+
|Braund, Mr. Owen ...|      Mr|22.0|       S|
|Cumings, Mrs. Joh...|     Mrs|38.0|       C|
|Heikkinen, Miss. ...|    Miss|26.0|       S|
|Futrelle, Mrs. Ja...|     Mrs|35.0|       S|
|Allen, Mr. Willia...|      Mr|35.0|       S|
+--------------------+--------+----+--------+
only showing top 5 rows


Les quatre modèles et la validation croisée relisent l'ensemble d'apprentissage : sans `cache()`, chaque ajustement relirait le CSV et rejouerait l'expression régulière, la séparation et la jointure. Comme une transformation, `cache()` ne calcule rien : les données sont gardées en mémoire à la première action.

In [32]:
entrainement = entrainement.cache()

## Apprentissage automatique avec la MLlib (facultatif)

### Le pipeline

Un `Pipeline` enchaîne les étapes de la préparation et le modèle. Son `fit` ajuste chaque étape sur l'ensemble d'apprentissage, dans l'ordre, et renvoie un `PipelineModel`, dont `transform` applique les mêmes étapes, déjà ajustées, à d'autres données :

- `StringIndexer` remplace chaque catégorie par un entier ;
- `OneHotEncoder` remplace cet entier par des indicatrices (une colonne à 0 ou 1 par catégorie), pour qu'un modèle linéaire ne lise pas un ordre entre les ports ou les civilités ;
- `VectorAssembler` réunit les caractéristiques en un seul vecteur, la colonne qu'attendent les modèles de la MLlib ;
- le classifieur.

In [33]:
categorielles = ["sex", "embarked", "civilite"]
numeriques = ["pclass", "age", "sib_sp", "parch", "fare", "taille_famille", "seul"]


def construire_pipeline(classifieur):
  return Pipeline(
    stages=[
      MF.StringIndexer(
        inputCols=categorielles,
        outputCols=[f"{colonne}_index" for colonne in categorielles],
      ),
      MF.OneHotEncoder(
        inputCols=[f"{colonne}_index" for colonne in categorielles],
        outputCols=[f"{colonne}_indicatrices" for colonne in categorielles],
      ),
      MF.VectorAssembler(
        inputCols=numeriques + [f"{colonne}_indicatrices" for colonne in categorielles],
        outputCol="caracteristiques",
      ),
      classifieur,
    ]
  )

### Évaluation

Deux mesures sur l'ensemble de test : l'exactitude, la part des passagers bien classés, et l'aire sous la courbe ROC (AUC), qui juge le classement des scores sans dépendre du seuil de décision. La référence est l'exactitude de la classe majoritaire : prédire que personne ne survit.

In [34]:
exactitude = MulticlassClassificationEvaluator(
  labelCol="survived", predictionCol="prediction", metricName="accuracy"
)
auc = BinaryClassificationEvaluator(labelCol="survived")


def evaluer(modele, nom):
  predictions = modele.transform(test)
  print(
    f"{nom} : exactitude {exactitude.evaluate(predictions):.2f},"
    f" AUC {auc.evaluate(predictions):.2f}"
  )


nb_test = test.count()
nb_morts = test.where(F.col("survived") == 0).count()
print(
  f"Classe majoritaire : exactitude {nb_morts / nb_test:.2f} sur {nb_test} passagers"
)

Classe majoritaire : exactitude 0.63 sur 188 passagers


### Régression logistique

In [35]:
modele_logistique = construire_pipeline(
  LogisticRegression(labelCol="survived", featuresCol="caracteristiques")
).fit(entrainement)
evaluer(modele_logistique, "Régression logistique")
modele_logistique.transform(test).select(
  "name", "survived", "prediction", "probability"
).show(5)

Régression logistique : exactitude 0.82, AUC 0.86
+--------------------+--------+----------+--------------------+
|                name|survived|prediction|         probability|
+--------------------+--------+----------+--------------------+
|    Moran, Mr. James|       0|       0.0|[0.90317896523785...|
|McCarthy, Mr. Tim...|       0|       0.0|[0.70104875409283...|
|Saundercock, Mr. ...|       0|       0.0|[0.90439068452074...|
|Andersson, Mr. An...|       0|       0.0|[0.99272352209283...|
|Vestrom, Miss. Hu...|       0|       1.0|[0.31592575490621...|
+--------------------+--------+----------+--------------------+
only showing top 5 rows


### Forêt aléatoire

In [36]:
modele_foret = construire_pipeline(
  RandomForestClassifier(labelCol="survived", featuresCol="caracteristiques")
).fit(entrainement)
evaluer(modele_foret, "Forêt aléatoire")

Forêt aléatoire : exactitude 0.82, AUC 0.87


### Arbres boostés par gradient

In [37]:
modele_gbt = construire_pipeline(
  GBTClassifier(labelCol="survived", featuresCol="caracteristiques", maxIter=10)
).fit(entrainement)
evaluer(modele_gbt, "Arbres boostés par gradient")

Arbres boostés par gradient : exactitude 0.80, AUC 0.83


### Séparateur à vaste marge

In [38]:
modele_svm = construire_pipeline(
  LinearSVC(labelCol="survived", featuresCol="caracteristiques")
).fit(entrainement)
evaluer(modele_svm, "Séparateur à vaste marge")

26/11/11 22:29:40 ERROR OWLQN: Failure! Resetting history: breeze.optimize.NaNHistory: 


Séparateur à vaste marge : exactitude 0.82, AUC 0.86


### Réglage par validation croisée

`CrossValidator` choisit les hyperparamètres sur l'ensemble d'apprentissage seul : il le découpe en trois, ajuste le pipeline sur deux tiers et le mesure sur le troisième, à tour de rôle, pour chaque combinaison de la grille que construit `ParamGridBuilder`, puis réajuste la meilleure sur tout l'apprentissage. Le test reste ainsi hors du choix. La grille porte ici sur la régularisation de la régression logistique.

In [39]:
logistique = LogisticRegression(labelCol="survived", featuresCol="caracteristiques")
grille = ParamGridBuilder().addGrid(logistique.regParam, [0.0, 0.1, 1.0]).build()
validation = CrossValidator(
  estimator=construire_pipeline(logistique),
  estimatorParamMaps=grille,
  evaluator=auc,
  numFolds=3,
  seed=11,
)
modele_valide = validation.fit(entrainement)
for parametres, auc_moyenne in zip(grille, modele_valide.avgMetrics):
  print(f"regParam = {parametres[logistique.regParam]} : AUC {auc_moyenne:.3f}")
meilleur_modele = modele_valide.bestModel
evaluer(meilleur_modele, "Régression logistique réglée")

regParam = 0.0 : AUC 0.873
regParam = 0.1 : AUC 0.870
regParam = 1.0 : AUC 0.857
Régression logistique réglée : exactitude 0.82, AUC 0.86


### Conclusion

Les quatre modèles dépassent nettement la classe majoritaire (0,63) : de 0,80 à 0,84 d'exactitude, de 0,83 à 0,88 d'AUC. Leurs écarts, eux, ne prouvent rien. Le score de la forêt aléatoire varie lui-même d'une exécution à l'autre (de 0,82 à 0,84 d'exactitude lors de nos essais), et le test ne compte que 188 passagers : chacun pèse un peu plus d'un demi-point d'exactitude, et la marge d'erreur d'une exactitude de 0,82 mesurée sur 188 passagers est d'environ 5 points de part et d'autre. Un autre découpage (une autre graine de `randomSplit`) pourrait changer le classement. Pour choisir entre des modèles ou des hyperparamètres, on compare donc leurs scores en validation croisée, sur l'apprentissage, comme `CrossValidator` le fait plus haut ; le test ne sert qu'à mesurer, une fois, le modèle retenu. Ici, la validation croisée préfère la régression logistique sans régularisation, le réglage par défaut : le modèle réglé est le premier.

## Sauvegarde et réutilisation du modèle

Un `PipelineModel` se sauvegarde avec toutes ses étapes ajustées (les catégories des encodeurs, les coefficients du modèle) : un autre programme Spark le recharge et l'applique à de nouvelles données, sans rien réapprendre. C'est ainsi qu'un modèle passe en production dans un traitement par lots.

In [40]:
chemin_modele = "modele_titanic"
meilleur_modele.write().overwrite().save(chemin_modele)
modele_charge = PipelineModel.load(chemin_modele)

Trois passagers inventés, décrits par les colonnes d'origine et leur civilité ; la taille de la famille et `seul` se calculent comme plus haut, le modèle rechargé fait le reste :

In [41]:
nouveaux_passagers = spark.createDataFrame(
  [
    ("female", 1, 35.0, 1, 0, 80.0, "C", "Mrs"),
    ("male", 3, 28.0, 0, 0, 8.0, "S", "Mr"),
    ("male", 2, 6.0, 1, 1, 30.0, "S", "Master"),
  ],
  ["sex", "pclass", "age", "sib_sp", "parch", "fare", "embarked", "civilite"],
)
nouveaux_passagers = nouveaux_passagers.withColumn(
  "taille_famille", F.col("sib_sp") + F.col("parch") + 1
).withColumn("seul", (F.col("taille_famille") == 1).cast("long"))
modele_charge.transform(nouveaux_passagers).select(
  "sex", "pclass", "age", "civilite", "probability", "prediction"
).show(truncate=False)

+------+------+----+--------+----------------------------------------+----------+
|sex   |pclass|age |civilite|probability                             |prediction|
+------+------+----+--------+----------------------------------------+----------+
|female|1     |35.0|Mrs     |[0.03789324277389312,0.9621067572261068]|1.0       |
|male  |3     |28.0|Mr      |[0.9213895603024594,0.07861043969754056]|0.0       |
|male  |2     |6.0 |Master  |[0.13784107749083357,0.8621589225091664]|1.0       |
+------+------+----+--------+----------------------------------------+----------+



## Le même code sur un cluster

Rien de ce notebook ne dépend du mode local, sauf la session. Sur un cluster, on la crée avec l'adresse de son gestionnaire, par exemple `SparkSession.builder.master("spark://hote:7077")` pour un cluster Spark autonome, ou bien `spark-submit --master yarn` lance le script ; les chemins désignent alors un stockage partagé (HDFS, S3…), lisible par tous les exécuteurs. Les partitions se répartissent entre les machines, et les étapes qui regroupent des lignes de partitions différentes (un `groupBy`, l'ajustement d'un modèle) font circuler des données entre elles : c'est l'étape `Exchange` qu'affiche `explain` sur un `groupBy`. Le code des transformations, du pipeline et de la sauvegarde, lui, ne change pas.

## Référence

https://spark.apache.org/docs/latest/ml-classification-regression.html